In [1]:
from BioRED_python_download.biored import BioredDataset
from transformers import DistilBertModel, DistilBertTokenizer, DistilBertForSequenceClassification, DataCollatorWithPadding
from nltk.tokenize import word_tokenize
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DistilBertModel, DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import AutoTokenizer
import numpy as np
from transformers import AutoModelForTokenClassification
from transformers import DataCollatorForTokenClassification
import itertools


# Select existing config
builder = BioredDataset(config_name="biored_bigbio_kb")

# Download and prepare the dataset
builder.download_and_prepare()

# Load dataset splits. That is, NO NEED FOR SEED because we are using pre-defined splits from BioRED
dataset = builder.as_dataset()

train = dataset["train"]
test = dataset["test"]
val = dataset["validation"]

print("Column names:\n", train.column_names)
print("Features:\n", train.features)
print("Entities:\n", train.features["entities"])
print("Number of columns:", train.num_columns)

print("First data sample in train:\n", train[0])

sample = train[0]
print(f"Title/Abstract: {sample['passages'][0]['text']}")
print(f"First Entity: {sample['entities'][0]}")
print(f"First Relation: {sample['relations'][0]}")

'''print("First data sample in train (line by line):")
print(train[0]["id"])
print(train[0]["document_id"])
print(train[0]["passages"])
print(train[0]["entities"])
print(train[0]["events"])
print(train[0]["coreferences"])
print(train[0]["relations"])'''

print("Size of training subset (i.e. number of data samples):", train.num_rows)
print("Size of validation subset (i.e. number of data samples):", val.num_rows)
print("Size of testing subset (i.e. number of data samples):", test.num_rows)


Column names:
 ['id', 'document_id', 'passages', 'entities', 'events', 'coreferences', 'relations']
Features:
 {'id': Value('string'), 'document_id': Value('string'), 'passages': List({'id': Value('string'), 'type': Value('string'), 'text': List(Value('string')), 'offsets': List(List(Value('int32')))}), 'entities': List({'id': Value('string'), 'type': Value('string'), 'text': List(Value('string')), 'offsets': List(List(Value('int32'))), 'normalized': List({'db_name': Value('string'), 'db_id': Value('string')})}), 'events': List({'id': Value('string'), 'type': Value('string'), 'trigger': {'text': List(Value('string')), 'offsets': List(List(Value('int32')))}, 'arguments': List({'role': Value('string'), 'ref_id': Value('string')})}), 'coreferences': List({'id': Value('string'), 'entity_ids': List(Value('string'))}), 'relations': List({'id': Value('string'), 'type': Value('string'), 'arg1_id': Value('string'), 'arg2_id': Value('string'), 'normalized': List({'db_name': Value('string'), 'db_

In [2]:
# 1. Initialize Tokenizer (BioBERT is standard for BioRED)
tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-uncased', add_prefix_space=True)

# Map BioRED entity types to ID integers
label_list = [ "O",
               "B-ChemicalEntity", "I-ChemicalEntity",
               "B-DiseaseOrPhenotypicFeature","I-DiseaseOrPhenotypicFeature",
               "B-GeneOrGeneProduct","I-GeneOrGeneProduct",
               "B-SequenceVariant","I-SequenceVariant",
               "B-OrganismTaxon","I-OrganismTaxon",
               "B-CellLine", "I-CellLine"
               ]#["O", "B-Gene", "I-Gene", "B-Disease", "I-Disease", "B-Chemical", "I-Chemical"]  # Simplified for example. #["GeneOrGeneProduct", "DiseaseOrPhenotypicFeature", "ChemicalEntity", "SequenceVariant", "OrganismTaxon", "CellLine"]
label_to_id = {l: i for i, l in enumerate(label_list)}

# Map BioRED relation types to ID integers
relation_list = [
    "None", #not in original BioRED dataset, but added for logic to determine label
    "Association",
    "Bind",
    "Comparison",
    "Conversion",
    "Cotreatment",
    "Drug_Interaction",
    "Negative_Correlation",
    "Positive_Correlation"
]
relation_to_id = {r: i for i, r in enumerate(relation_list)}
id_to_relation = {i: r for i, r in enumerate(relation_list)}

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

def preprocess_biored_for_ner(examples):
    # BioRED KB has 'passages' containing the full text
    texts = [
        " ".join(
            passage_text
            for passage in p
            for passage_text in passage["text"]
        )
        for p in examples["passages"]
    ]

    tokenized_inputs = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=512,
        return_offsets_mapping=True  # CRITICAL for alignment
    )

    labels = []
    for i, document_entities in enumerate(examples["entities"]):
        offset_mapping = tokenized_inputs["offset_mapping"][i]
        doc_labels = np.zeros(len(offset_mapping), dtype=int)

        for entity in document_entities:
            start_char = entity["offsets"][0][0]
            end_char = entity["offsets"][0][1]
            entity_type = entity["type"]
            if entity_type not in [
                "ChemicalEntity",
                "DiseaseOrPhenotypicFeature",
                "GeneOrGeneProduct",
                "SequenceVariant",
                "OrganismTaxon",
                "CellLine"
            ]:
                print(entity_type + " " + str(start_char) + " " + str(end_char))

            # Map tokens to labels based on character offsets
            inside = False
            for idx, (start, end) in enumerate(offset_mapping):
                if start is None or start == end: #if start == end:  # Skip special tokens like [CLS] or [PAD]
                    doc_labels[idx] = -100  # Standard ignore index for loss
                    continue

                if not (end <= start_char or start >= end_char): #if start >= start_char and end <= end_char
                    # Logic for B- and I- tags
                    if not inside:
                        prefix = "B-"
                        inside = True
                    else:
                        prefix = "I-"
                    label_str = f"{prefix}{entity_type}"

                    if label_str in label_to_id:
                        doc_labels[idx] = label_to_id[label_str]
                    else:
                        raise ValueError(f"Unknown label: {label_str}")
        labels.append(doc_labels.tolist())

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


def preprocess_biored_for_re(example):
    entities = example["entities"]
    existing_relations = {(r["arg1_id"], r["arg2_id"]): r["type"] for r in example["relations"]}

    for r in example["relations"]:
        #print(r['type'])
        if r['type'] not in relation_list:
            print("RELATION LIST TEST, relation missing:\n", r, "\n ", r['type'])

    #Alternative to be tried out if for training current implementation of method returning a dictionary in the way it currently does (even though as per HuggingFace it does need to return a dictionary) causes problems
    texts = []
    labels = []

    context = " ".join(example["passages"][0]["text"]) #context = " ".join(example["passages"][0]["text"])  # Simple abstract-level context
    # Generate every possible combination of 2 entities in the document
    for ent1, ent2 in itertools.combinations(entities, 2):
        pair = (ent1["id"], ent2["id"])
        reverse_pair = (ent2["id"], ent1["id"])

        # Determine label (Positive relation or 'None' for Negative Sampling)
        relation_type = "None"
        if pair in existing_relations:
            relation_type = existing_relations[pair]
        elif reverse_pair in existing_relations:
            relation_type = existing_relations[reverse_pair]

        # Create a text representation with Entity Markers and feed entity pair as a sentence pair to the tokenizer
        # "Gene [E1] Factor-6 [/E1] is linked to [E2] Diabetes [/E2]"
        entity_pair_str = f"{ent1['text'][0]} [SEP] {ent2['text'][0]}"

        texts.append((context, entity_pair_str))  # pair-level tokenization, unlike for NER, which requires document-level tokenization and hence, why input is tokenized at a different stage / before preparing the data for the NER module
        if relation_type not in relation_to_id:
            raise ValueError(f"Unknown relation type: {relation_type}")
        labels.append(relation_to_id[relation_type])  # e.g. 1

        # Tokenize all pairs for this document at once
    tokenized = tokenizer(
        [t[0] for t in texts],   # contexts
        [t[1] for t in texts],   # entity pairs
        truncation=True,
        padding="max_length",
        max_length=512
    )

    tokenized["labels"] = labels
    return tokenized


In [3]:
if __name__ == "__main__":
    train_ner = train.map(preprocess_biored_for_ner, batched=True) # apply to train subset
    val_ner = val.map(preprocess_biored_for_ner, batched=True)  # apply to val subset
    test_ner = test.map(preprocess_biored_for_ner, batched=True)  # apply to test subset

    #print(train_ner[0])
    #print("NER data preprocessing complete")

    train_re = train.map(preprocess_biored_for_re) #, batched=True) # apply to train subset
    val_re = val.map(preprocess_biored_for_re) #, batched=True)  # apply to val subset
    test_re = test.map(preprocess_biored_for_re) #, batched=True)  # apply to test subset

    #print(train_re[0])
    #print("RE data preprocessing complete")

    train_ner.set_format(type="torch") # convert it to tensor for torch and Transformers to be able to handle the training subset for NER
    val_ner.set_format(type="torch") # convert it to tensor for torch and Transformers to be able to handle the val subset for NER
    test_ner.set_format(type="torch") # convert it to tensor for torch and Transformers to be able to handle the test subset for NER

    train_re.set_format(type="torch") # convert it to tensor for torch and Transformers to be able to handle the training subset for RE
    val_re.set_format(type="torch") # convert it to tensor for torch and Transformers to be able to handle the val subset for RE
    test_re.set_format(type="torch") # convert it to tensor for torch and Transformers to be able to handle the test subset for RE

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]